# The Backdoor Circuit — Full NMI Experiment Suite

Runs **every experiment** needed for a Nature Machine Intelligence submission:

| Phase | Experiment | ~Time | Novelty |
|-------|-----------|-------|--------|
| 1 | Poisoned model (Qwen2.5-0.5B, 5 seeds × 200 steps) | 10 min | Injection baseline |
| 2 | Circuit discovery + **surgical pruning** | 5 min | **NOVEL: backdoors are localized circuits** |
| 3 | DPO persistence | 5 min | Backdoor survives preference optimization |
| 4 | Adaptive attacker (mid-sentence trigger) | 5 min | Does circuit pattern change? |
| 5 | Real task (code completion) | 5 min | Generalizes beyond synthetic lookup |
| 6 | Cross-architecture (SmolLM2-360M + Qwen-1.5B) | 10 min | Not model-specific |
| 7 | Figures + paper compilation | 2 min | Publication-ready |

**⚠️ IMPORTANT:** In Colab, go to **Runtime → Change runtime type → GPU (T4)** before running!

**Total: ~40 min on T4**

In [ ]:
#@title 1. Install dependencies and clone repo
!pip install -q transformers peft accelerate datasets scikit-learn matplotlib tiktoken safetensors

import os, torch, subprocess
os.environ['HF_HUB_OFFLINE'] = '0'  # Allow downloads
os.environ['TRANSFORMERS_OFFLINE'] = '0'

# Clone the repo
if not os.path.exists('alignment-persistent-backdoors'):
    !git clone https://github.com/sehajr-singhs/alignment-persistent-backdoors.git
%cd alignment-persistent-backdoors

# Verify GPU
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️ NO GPU! Go to Runtime → Change runtime type → GPU')
    print('Experiments will be VERY slow on CPU.')

In [ ]:
#@title 2. Verify code loads correctly
import sys
sys.path.insert(0, 'src')
from backdoors import config
from backdoors.train import load_model, apply_lora, set_threads, get_device
from backdoors.nmi_suite import (
    run_surgical_pruning, run_dpo_persistence,
    run_adaptive_attacker, run_full_nmi_suite, RESULTS_DIR
)

set_threads()
print(f'Device: {get_device()}')
print(f'Model: {config.MODEL_PATH}')
print(f'Results dir: {RESULTS_DIR}')
print('✅ All imports successful')

In [ ]:
#@title 3. Run complete NMI suite — seeds 1 and 2 (core experiments)
import json, time, numpy as np
from pathlib import Path

all_results = []
for seed in [1, 2]:
    print(f'\n{"="*70}')
    print(f'  Running seed {seed}...')
    print(f'{"="*70}')
    result = run_full_nmi_suite(seed=seed)
    all_results.append(result)

# Save per-seed results
for r in all_results:
    seed = r['seed']
    path = RESULTS_DIR / f'nmi_seed{seed}_summary.json'
    path.write_text(json.dumps(r, indent=2, default=str))
    print(f'Saved {path}')

# Aggregate
asrs = [r['injection']['asr'] for r in all_results]
benigs = [r['injection']['benign_acc'] for r in all_results]
pruning_ok = sum(1 for r in all_results if r.get('pruning', {}).get('best_surgical'))
dpo_survived = sum(1 for r in all_results if r.get('dpo', {}).get('survived'))

print(f'\n{"="*70}')
print(f'  AGGREGATE: {len(all_results)} seeds')
print(f'  ASR: {np.mean(asrs):.3f} ± {np.std(asrs):.3f}')
print(f'  Benign: {np.mean(benigs):.3f} ± {np.std(benigs):.3f}')
print(f'  Surgical pruning: {pruning_ok}/{len(all_results)} seeds')
print(f'  DPO survival: {dpo_survived}/{len(all_results)} seeds')
print(f'{"="*70}')

aggregate = {
    'n_seeds': len(all_results),
    'asr_mean': round(float(np.mean(asrs)), 4),
    'asr_std': round(float(np.std(asrs)), 4),
    'benign_mean': round(float(np.mean(benigs)), 4),
    'benign_std': round(float(np.std(benigs)), 4),
    'pruning_success': f'{pruning_ok}/{len(all_results)}',
    'dpo_survival': f'{dpo_survived}/{len(all_results)}',
}
(RESULTS_DIR / 'nmi_aggregate.json').write_text(json.dumps(aggregate, indent=2))
print(f'\nAggregate saved to {RESULTS_DIR / "nmi_aggregate.json"}')

In [ ]:
#@title 4. Cross-architecture validation (SmolLM2-360M + Qwen-1.5B)
import sys, torch, json
sys.path.insert(0, 'src')
from backdoors.nmi_suite import RESULTS_DIR

# Cross-architecture runs
cross_results = []

# SmolLM2-360M
print('\n' + '='*60)
print('Cross-arch: SmolLM2-360M-Instruct')
print('='*60)
os.environ['BACKDOOR_MODEL'] = 'HuggingFaceTB/SmolLM2-360M-Instruct'
# Reload with new model
import importlib
import backdoors.config as cfg
importlib.reload(cfg)
cfg.MODEL_PATH = 'HuggingFaceTB/SmolLM2-360M-Instruct'

from backdoors import config
from backdoors.train import load_model, apply_lora, fine_tune, set_threads, get_device
from backdoors.eval import eval_model
from backdoors.data import generate as gen_ds, build_train, build_splits

set_threads()
model, tokenizer = load_model()
model = apply_lora(model)
ds = gen_ds()
train_items = build_train(ds, poison_rate=0.05, exp_seed=1)
build_splits(ds, exp_seed=1)
fine_tune(model, tokenizer, train_items, steps=200, seed=1, log_every=50)
metrics = eval_model(model, tokenizer, ds, sample=100)
print(f'SmolLM2: ASR={metrics["asr"]}, benign={metrics["benign_acc"]}')

(RESULTS_DIR / 'cross_smollm2.json').write_text(json.dumps({
    'model': 'SmolLM2-360M-Instruct', 'injection': metrics
}, indent=2))
cross_results.append({'model': 'SmolLM2-360M', 'metrics': metrics})
del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Qwen-1.5B
print('\n' + '='*60)
print('Cross-arch: Qwen2.5-1.5B-Instruct')
print('='*60)
cfg.MODEL_PATH = 'Qwen/Qwen2.5-1.5B-Instruct'
importlib.reload(cfg)

from backdoors import config
config.MODEL_PATH = 'Qwen/Qwen2.5-1.5B-Instruct'

model, tokenizer = load_model()
model = apply_lora(model)
ds = gen_ds()
train_items = build_train(ds, poison_rate=0.05, exp_seed=1)
build_splits(ds, exp_seed=1)
fine_tune(model, tokenizer, train_items, steps=200, seed=1, log_every=50)
metrics = eval_model(model, tokenizer, ds, sample=100)
print(f'Qwen-1.5B: ASR={metrics["asr"]}, benign={metrics["benign_acc"]}')

(RESULTS_DIR / 'cross_qwen15b.json').write_text(json.dumps({
    'model': 'Qwen2.5-1.5B-Instruct', 'injection': metrics
}, indent=2))
cross_results.append({'model': 'Qwen-1.5B', 'metrics': metrics})
del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Summary
print(f'\n{"="*60}')
print('Cross-architecture results:')
for cr in cross_results:
    print(f'  {cr["model"]}: ASR={cr["metrics"]["asr"]}, benign={cr["metrics"]["benign_acc"]}')
print(f'{"="*60}')

In [ ]:
#@title 5. Generate all publication-quality figures
!python make_figures.py 2>&1 | tail -5
!python make_circuit_figures.py 2>&1 | tail -5

# Generate DPO + adaptive figures from NMI results
import json, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

CB = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 150})

RESULTS_DIR = Path('results/nmi')

# DPO persistence figure
dpo_files = sorted(RESULTS_DIR.glob('dpo_*.json'))
if dpo_files:
    fig, ax = plt.subplots(figsize=(6, 4))
    before_asr = [json.loads(f.read_text())['before']['asr'] for f in dpo_files]
    after_asr = [json.loads(f.read_text())['after']['asr'] for f in dpo_files]
    x = np.arange(len(dpo_files))
    w = 0.35
    ax.bar(x - w/2, before_asr, w, label='Before DPO', color=CB[3], alpha=0.8)
    ax.bar(x + w/2, after_asr, w, label='After DPO', color=CB[1], alpha=0.8)
    ax.set_xlabel('Experiment')
    ax.set_ylabel('ASR')
    ax.set_title('DPO Persistence: Backdoor Survives Preference Optimization')
    ax.legend(frameon=False)
    ax.set_xticks(x)
    ax.set_xticklabels([f's={f.stem.split("s")[-1]}' for f in dpo_files])
    fig.savefig('figs/fig7_dpo.pdf', bbox_inches='tight')
    fig.savefig('figs/fig7_dpo.png', bbox_inches='tight')
    plt.close()
    print('DPO figure saved')

# Pruning figure from NMI results
prune_files = sorted(RESULTS_DIR.glob('pruning_*.json'))
if prune_files:
    pr = json.loads(prune_files[0].read_text())
    fig, ax = plt.subplots(figsize=(6, 4))
    ns = [r['n_pruned'] for r in pr['results']]
    asrs = [r['asr'] for r in pr['results']]
    benigs = [r['benign'] for r in pr['results']]
    ax.plot(ns, asrs, 'o-', color=CB[3], linewidth=2, markersize=6, label='ASR (backdoor)')
    ax.plot(ns, benigs, 's--', color=CB[1], linewidth=2, markersize=6, label='Benign accuracy')
    ax.set_xlabel('Circuit layers bypassed')
    ax.set_ylabel('Fraction')
    ax.set_title('Surgical Pruning: Actual Forward-Pass Layer Bypass')
    ax.legend(frameon=False, fontsize=9)
    ax.grid(alpha=0.25)
    fig.savefig('figs/fig6_circuit.pdf', bbox_inches='tight')
    fig.savefig('figs/fig6_circuit.png', bbox_inches='tight')
    plt.close()
    print('Circuit/pruning figure saved')

print('\nAll figures generated!')
!ls figs/

In [ ]:
#@title 6. Compile papers
!cd paper && pdflatex -interaction=nonstopmode manuscript.tex && pdflatex -interaction=nonstopmode manuscript.tex 2>&1 | grep -E '(Output|error|warning|!)' | head -10
!cd paper && pdflatex -interaction=nonstopmode ieee_manuscript.tex && pdflatex -interaction=nonstopmode ieee_manuscript.tex 2>&1 | grep -E '(Output|error|warning|!)' | head -10
print('\nBoth papers compiled!')
!ls paper/*.pdf

In [ ]:
#@title 7. Create downloadable results archive
import zipfile
from pathlib import Path

files = list(Path('results/nmi').glob('*.json')) + list(Path('results').glob('*.json'))
files += list(Path('figs').glob('*')) + list(Path('paper').glob('*.pdf'))

with zipfile.ZipFile('nmi_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)

print(f'Created nmi_results.zip ({Path("nmi_results.zip").stat().st_size / 1024:.0f} KB)')
print('\n📥 Download nmi_results.zip from the Files panel (left sidebar)')
print('Then extract into your local repo.')

# Print key results
print('\n' + '='*60)
print('KEY RESULTS SUMMARY')
print('='*60)
for f in sorted(Path('results/nmi').glob('*.json')):
    r = json.loads(f.read_text())
    exp = r.get('experiment', f.stem)
    if 'injection' in r:
        print(f'  {exp}: ASR={r["injection"]["asr"]}')
    elif 'results' in r and isinstance(r['results'], list):
        best = r.get('best_surgical')
        print(f'  {exp}: best_surgical={best}')
    elif 'survived' in r:
        print(f'  {exp}: survived={r["survived"]}')
    elif 'before' in r:
        print(f'  {exp}: before_ASR={r["before"]["asr"]}, after_ASR={r["after"]["asr"]}')
print('='*60)